# ONNX Runtime CPU EP Performance Analysis

**Objective:** Compare ORT CPU EP latency across quantization formats (MNB native, QDQ fused, QDQ unfused), weight layouts (original vs transposed), bit widths (4, 8), symmetry/signedness/granularity variants across devices (AMD, Intel, ARM).

**Devices:**
| Label | CPU | Notes |
|-------|-----|-------|
| `amd` | AMD64 Family 26 Model 36 (HP AMD laptop) | ORT 1.24.2, 10 iterations |
| `intel` | Intel64 Family 6 Model 186 (Surface Laptop) | ORT 1.24.1, 10 iterations |
| `arm` | ARM64 (Surface ARM laptop) | ORT 1.24.1, 50 iterations |

**Notes:**
- ORT only fuses DQ+MatMul → MatMulNBits for **4-bit block** quantization 
- Transpose models add a Transpose between DQ and MatMul to test column-major weight layout
- 8 models failed to load for `channel-sym-signed-transpose` variants (possible ORT bug)
- ARM data seem to have higher variance and maybe some memory pressure issues for 7b-8 bit models. Might need to examine/rerun ARM test

In [89]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

# --- Load all three devices ---
DATA_DIR = Path(r"C:\dev\cpu_perf")
DEVICES = {
    "amd": DATA_DIR / "hp_amd_laptop_perf",
    "intel": DATA_DIR / "surface_laptop_perf",
    "arm": DATA_DIR / "surface_arm_laptop_perf",
}

frames = []
for device, path in DEVICES.items():
    csv_path = path / "summary.csv"
    if not csv_path.exists():
        print(f"WARNING: {csv_path} not found, skipping")
        continue
    tmp = pd.read_csv(csv_path)
    tmp["device"] = device
    frames.append(tmp)
    print(f"  {device}: {len(tmp)} rows from {csv_path.name}")

df = pd.concat(frames, ignore_index=True)

# Type coercion
df["bits"] = df["bits"].astype(int)
df["seq_length"] = df["seq_length"].astype(int)
df["mean_ms"] = df["mean_ms"].astype(float)
df["model_load_time_s"] = pd.to_numeric(df["model_load_time_s"], errors="coerce")

print(f"\nTotal rows: {len(df)}")
print(f"Devices: {list(df['device'].unique())}")
print(f"Weight layouts: {list(df['weight_layout'].unique())}")
print(f"Scenarios: {list(df['scenario'].unique())}")
print(f"Seq lengths: {sorted(int(x) for x in df['seq_length'].unique())}")
print(f"Bits: {sorted(int(x) for x in df['bits'].unique())}")
print(f"Model sizes: {list(df['model_size'].unique())}")

  amd: 640 rows from summary.csv
  intel: 640 rows from summary.csv
  arm: 560 rows from summary.csv

Total rows: 1840
Devices: ['amd', 'intel', 'arm']
Weight layouts: ['original', 'transpose']
Scenarios: ['native', 'qdq_fused', 'qdq_unfused']
Seq lengths: [1, 128, 256, 512, 1024]
Bits: [4, 8]
Model sizes: ['0.5b', '1.5b', '3b', '7b']


In [59]:
# Save CSV
out_path = Path("all_devices_summary.csv")
mnb_mask = df["format_type"] == "mnb"
df.loc[mnb_mask, "granularity"] = "n/a"
df.loc[mnb_mask, "signedness"] = "n/a"
print(f"Set {mnb_mask.sum()} MNB rows: granularity='n/a', signedness='n/a'")

# Re-save CSV
df.to_csv(out_path, index=False)
print(f"Updated {out_path}")

Set 240 MNB rows: granularity='n/a', signedness='n/a'
Updated all_devices_summary.csv


In [90]:
# --- Helper functions ---
MODEL_SIZE_ORDER = ["0.5b", "1.5b", "3b", "7b"]
SEQ_ORDER = [1, 128, 256, 512, 1024]
DEVICE_ORDER = ["amd", "intel", "arm"]

def pivot_latency(data, index_cols, columns_col, values_col="mean_ms"):
    """Pivot data into a wide table with mean_ms values."""
    return data.pivot_table(
        index=index_cols, columns=columns_col,
        values=values_col, aggfunc="first"
    )


def ratio_table(data, col_a, col_b, label_a="A", label_b="B"):
    """Compute ratio B/A for two subsets identified by a column value."""
    a = data[data[col_a] == label_a].copy()
    b = data[data[col_a] == label_b].copy()
    merge_cols = [c for c in a.columns if c not in ["mean_ms", col_a, "model_load_time_s"]]
    merged = a.merge(b, on=merge_cols, suffixes=(f"_{label_a}", f"_{label_b}"))
    merged["ratio"] = merged[f"mean_ms_{label_b}"] / merged[f"mean_ms_{label_a}"]
    return merged


def comparison_table(data, group_cols, compare_col, compare_values, metric="mean_ms"):
    """Build a side-by-side comparison table.

    Returns a DataFrame with one row per (group_cols) combination and
    one column per compare_value, plus a ratio column if exactly 2 compare_values.
    """
    tables = {}
    for val in compare_values:
        subset = data[data[compare_col] == val].set_index(group_cols)[metric]
        tables[val] = subset

    result = pd.DataFrame(tables)
    if len(compare_values) == 2:
        a, b = compare_values
        result[f"{b}/{a}"] = result[b] / result[a]
    return result


def display_comparison(title, data, group_cols, compare_col, compare_values, metric="mean_ms"):
    """Print a comparison table with a title."""
    print(f"\n{'='*80}")
    print(f"  {title}")
    print(f"{'='*80}")
    tbl = comparison_table(data, group_cols, compare_col, compare_values, metric)
    print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))
    return tbl



## A. MNB Native vs QDQ Fused vs QDQ Unfused — 4-bit block quantization



**Expected**: MNB ≈ QDQ fused << QDQ unfused


In [68]:

# Filter to 4-bit, block-quantized, original layout, asym signed (most common LLM config)
mnb_4 = df[(df["format_type"] == "mnb") & (df["bits"] == 4) & (df["weight_layout"] == "original")].copy()
qdq_fused_4 = df[
    (df["format_type"] == "qdq") & (df["bits"] == 4) & (df["granularity"] == "block") &
    (df["weight_layout"] == "original") & (df["scenario"] == "qdq_fused") & (df["signedness"] == "signed")
].copy()
qdq_unfused_4 = df[
    (df["format_type"] == "qdq") & (df["bits"] == 4) & (df["granularity"] == "block") &
    (df["weight_layout"] == "original") & (df["scenario"] == "qdq_unfused") & (df["signedness"] == "signed")
].copy()

# Build merged comparison table (asym only — sym behaves identically)
mnb_sub = mnb_4[mnb_4["symmetry"] == "asym"][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "mnb_ms"})
fused_sub = qdq_fused_4[qdq_fused_4["symmetry"] == "asym"][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "fused_ms"})
unfused_sub = qdq_unfused_4[qdq_unfused_4["symmetry"] == "asym"][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "unfused_ms"})

merged = mnb_sub.merge(fused_sub, on=["device", "model_size", "seq_length"], how="outer")
merged = merged.merge(unfused_sub, on=["device", "model_size", "seq_length"], how="outer")
merged["fused/mnb"] = merged["fused_ms"] / merged["mnb_ms"]
merged["unfused/mnb"] = merged["unfused_ms"] / merged["mnb_ms"]
merged = merged.set_index(["device", "model_size", "seq_length"]).sort_index()

# --- Table 1: Latency (ms) — seq=1 (decode) and seq=1024 (prefill) ---
print("  Table 1: Latency comparison (ms) — 4-bit block asym signed, original layout")
print("  " + "-"*85)
for seq in [1, 1024]:
    label = "Decode (seq=1)" if seq == 1 else "Prefill (seq=1024)"
    sub = merged.xs(seq, level="seq_length")[["mnb_ms", "fused_ms", "unfused_ms", "fused/mnb", "unfused/mnb"]]
    print(f"\n  {label}:")
    print(sub.to_string(float_format=lambda x: f"{x:.1f}"))

# --- Table 2: Model load times ---
mnb_load = mnb_4[mnb_4["symmetry"] == "asym"].drop_duplicates(subset=["device", "model_size"])[["device", "model_size", "model_load_time_s"]].rename(columns={"model_load_time_s": "mnb_s"})
fused_load = qdq_fused_4[qdq_fused_4["symmetry"] == "asym"].drop_duplicates(subset=["device", "model_size"])[["device", "model_size", "model_load_time_s"]].rename(columns={"model_load_time_s": "fused_s"})
unfused_load = qdq_unfused_4[qdq_unfused_4["symmetry"] == "asym"].drop_duplicates(subset=["device", "model_size"])[["device", "model_size", "model_load_time_s"]].rename(columns={"model_load_time_s": "unfused_s"})
load_times = mnb_load.merge(fused_load, on=["device", "model_size"]).merge(unfused_load, on=["device", "model_size"])
load_times["fused/mnb"] = load_times["fused_s"] / load_times["mnb_s"]

print(f"\n\n  Table 2: Model load times (s) — includes PrePack for MNB & fused; unfused skips PrePack")
print("  " + "-"*85)
print(load_times.set_index(["device", "model_size"]).sort_index().to_string(float_format=lambda x: f"{x:.2f}"))


  Table 1: Latency comparison (ms) — 4-bit block asym signed, original layout
  -------------------------------------------------------------------------------------

  Decode (seq=1):
                   mnb_ms  fused_ms  unfused_ms  fused/mnb  unfused/mnb
device model_size                                                      
amd    0.5b           7.1       8.3       691.9        1.2         97.2
       1.5b          19.2      20.6      2267.2        1.1        118.3
       3b            46.6      41.0      4388.0        0.9         94.2
       7b            98.1      78.8      8121.9        0.8         82.8
arm    0.5b           9.7      11.3       624.2        1.2         64.4
       1.5b          87.9      33.6      1792.3        0.4         20.4
       3b           116.5      56.4      3461.7        0.5         29.7
       7b           112.0      59.1     10612.2        0.5         94.8
intel  0.5b           8.9       8.4       984.2        0.9        110.7
       1.5b          22

In [69]:

# --- Table 3: Estimated DequantizeLinear overhead (unfused - fused) at seq=1 ---
# At seq=1 the MatMul compute is trivial, so the delta ≈ pure DQ cost
m = merged.copy()
m["dq_overhead_ms"] = m["unfused_ms"] - m["fused_ms"]

seq1 = m.xs(1, level="seq_length")[["fused_ms", "unfused_ms", "dq_overhead_ms"]]

print("  Table 3: Estimated DequantizeLinear cost at seq=1 (ms)")
print("  At decode (seq=1), MatMul compute is trivial → delta ≈ pure DQ overhead")
print("  " + "-"*70)
print(seq1.to_string(float_format=lambda x: f"{x:.0f}"))

print("\n\n  DQ cost scaling relative to 0.5b (should be ~linear with param count):")
for dev in DEVICE_ORDER:
    dd = seq1.xs(dev, level="device")["dq_overhead_ms"]
    base = dd.iloc[0]
    print(f"    {dev}: " + "  ".join(f"{sz}={v/base:.1f}x" for sz, v in dd.items()))


  Table 3: Estimated DequantizeLinear cost at seq=1 (ms)
  At decode (seq=1), MatMul compute is trivial → delta ≈ pure DQ overhead
  ----------------------------------------------------------------------
                   fused_ms  unfused_ms  dq_overhead_ms
device model_size                                      
amd    0.5b               8         692             684
       1.5b              21        2267            2247
       3b                41        4388            4347
       7b                79        8122            8043
arm    0.5b              11         624             613
       1.5b              34        1792            1759
       3b                56        3462            3405
       7b                59       10612           10553
intel  0.5b               8         984             976
       1.5b              23        3190            3168
       3b                44        6257            6212
       7b               101       14140           14039


  DQ cost 

### Observations

**1. Fused QDQ ≈ MNB Native — same kernel, confirmed in source**

Both paths run `MatMulNBits` → `MlasQNBitGemmBatch()`. Verified: same `accuracy_level=4`, same PrePack, same MLAS kernel.
- **Intel**: fused/mnb ≈ 0.94–1.17×  as expected
- **AMD**: small models slightly slower (0.5b: ~1.2×), large models slightly faster (3b–7b: ~0.9×) 
- **ARM**: fused consistently faster (0.4–0.8×), MNB slow at seq=1 for ≥1.5b (might need a rerun)

**2. Unfused QDQ is 80–150× slower at decode**

The bottleneck seem to be `DequantizeLinear` for 4-bit sub-byte types. The kernel (`quantize_linear.cc:451`)

```cpp
// Line 449: TODO(fajin) : add mlas kernel to utilize multithreading, refer MlasDequantizeBlockwise.
ORT_UNUSED_PARAMETER(thread_pool);  // ← thread pool ignored, runs single-threaded
...
for (size_t bs = 0; bs < N; ++bs, ++input_index) {
    int32_t val = static_cast<int32_t>(input[input_index >> shift_bits].GetElem(input_index & mask));
    *output++ = static_cast<OutT>(static_cast<float>(val - zp) * sc);
}
```
The function is single threaded and `ctx->Output(0, x_shape)` at line 547 allocates full `[K, N]` float32 just for the subsequent `MatMul` to consume


**3. Load times**

Fused QDQ loads 1.3–1.6× slower than MNB (extra weight transpose at graph optimization time). Unfused loads ~10× faster (skips PrePack).


---

#### I asked copilot to look deeper into itidentified following Optimization Opportunities for the Unfused DequantizeLinear Path

Optimization 1 (easy): Add threading to the blocked sub-byte path

The non-sub-byte DQ already has threading (gated behind `ORT_CLIENT_PACKAGE_BUILD` at line 340):
```cpp
ParDequantizeLinearStd<T>(input, output, N, scale[k], zero_point ? zero_point[k] : 0, thread_pool);
```
This calls `MlasTryBatchParallel` internally to split N elements across threads. But the blocked sub-byte path at line 451 explicitly ignores the thread pool.

**What to do**: Replace `ORT_UNUSED_PARAMETER(thread_pool)` at line 452 with `concurrency::ThreadPool::TryParallelFor` to partition the outer `M × num_blocks` loop across threads. The inner loop body has no data dependencies between blocks, so this is safe. This is the single most impactful change — it immediately scales with available cores.

Expected impact: **~4–8× speedup on multi-core systems** (proportional to core count).

Optimization 2 (medium): Call `MlasDequantizeBlockwise` (it already exists and is threaded)

MLAS already has a threaded dequantization function at `core/mlas/lib/q4_dq.cpp:1847`:
```cpp
template <typename ElementT, int qbits>
void MlasDequantizeBlockwise(ElementT* dst, const uint8_t* src, const ElementT* scales,
    const uint8_t* zero_points, int block_size, bool columnwise,
    int rows, int columns, MLAS_THREADPOOL* thread_pool);
```

`MatMulNBits::ComputeBUnpacked()` already calls it at `matmul_nbits.cc:599`:
```cpp
MlasDequantizeBlockwise<float, 4>(tmp_b_data_ptr.get(), b_data, scales_data,
    static_cast<const uint8_t*>(zero_points_data),
    static_cast<int32_t>(block_size_), column_wise_quant_,
    static_cast<int32_t>(K_), static_cast<int32_t>(N_), thread_pool);
```

**Blocker**: Layout mismatch. `MlasDequantizeBlockwise` expects **column-major** packed weights (for the MatMulNBits weight format). DequantizeLinear uses **row-major** packed weights (ONNX QDQ spec). From `q4_dq.cpp:391`: `"Source is row major. Dest, scale and zp are column major."`

**What to do**: Write a new MLAS function `MlasDequantizeBlockwiseRowMajor()` that keeps the threading (`MlasTryBatchParallel`) but adds SIMD nibble extraction for row-major layout. The SIMD patterns already exist in ORT:
- **AVX2** (`sqnbitgemm_kernel_avx2.cpp:265`): `_mm256_and_si256(data, _mm256_set1_epi8(0xF))` — extracts 32 nibbles per instruction
- **NEON** (`sqnbitgemm_kernel_neon_fp32.cpp:156`): `vand_u8(data, vdup_n_u8(0x0F))` + `vshr_n_u8(data, 4)` — 8+8 nibbles per instruction

Expected impact: **Another 4–8× on top of threading** from SIMD (32 elements/cycle on AVX2 vs 1 scalar).

Optimization 3 (medium): Add `PrePack()` for constant weight caching

`DequantizeLinear` has **no `PrePack()`** (it's a plain `OpKernel`, line 18). Every inference call re-dequantizes constant weights from scratch. Meanwhile, `MatMulNBits` has a full `PrePack()` at `matmul_nbits.cc:188` that pre-packs weights at load time.

**What to do**: Add a `PrePack()` override to `DequantizeLinear` that:
1. Checks if inputs 0 (data), 1 (scale), 2 (zero_point) are constant initializers
2. If yes, dequantizes once at load time and caches the float32 result
3. In `Compute()`, just `memcpy` the cached buffer to the output

For LLM weight dequantization, all inputs are always constant. This eliminates **100% of the per-inference DQ cost** — the only remaining cost is the memcpy.

Expected impact: **Eliminates repeat dequant entirely** (~10ms → ~1ms per DQ node from pure memcpy). Trade-off: higher memory usage (stores the full fp32 weights alongside int4).

Optimization 4 (easy): Remove the `ORT_CLIENT_PACKAGE_BUILD` gate

The non-sub-byte per-axis DQ threading/SIMD at line 340 is gated behind `#if defined(ORT_CLIENT_PACKAGE_BUILD)` with a TODO: *"Make this the default behavior after more testing."* 

This means that standard ORT builds don't get threading for int8/uint8 DequantizeLinear either. Removing this gate (after validation) would benefit all DQ users.

#### Summary: Effort vs Impact

| Optimization | Effort | Expected Speedup | File(s) |
|---|---|---|---|
| 1. Add threading | Low | ~4–8× | `quantize_linear.cc:451` |
| 2. SIMD nibble extraction | Medium | ~4–8× (on top of #1) | New MLAS fn + `quantize_linear.cc` |
| 3. PrePack for const weights | Medium | Eliminates repeat DQ | `quantize_linear.cc:18` |
| 4. Remove client build gate | Low | Enables int8 DQ threading | `quantize_linear.cc:340` |
| **1+2 combined** | **Medium** | **~16–60×** | — |

Optimizations 1+2 together could close most of the 80–150× gap vs fused. Optimization 3 would eliminate the gap entirely for constant weights (the common LLM case).


## B. MNB Symmetric vs Asymmetric

Check if asymmetric (with zero-point) adds overhead vs symmetric.

In [77]:
mnb = df[(df["format_type"] == "mnb") & (df["weight_layout"] == "original")].copy()

for bits in [4, 8]:
    sub = mnb[mnb["bits"] == bits]
    tbl = comparison_table(
        sub,
        group_cols=["device", "model_size", "seq_length"],
        compare_col="symmetry",
        compare_values=["sym", "asym"],
    )
    tbl.columns = ["sym", "asym", "asym/sym"]

    print(f"\n{'='*90}")
    print(f"  B. MNB sym vs asym — {bits}-bit (ratio = asym/sym)")
    print(f"{'='*90}")
    print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))

    # Summary stats per device
    print(f"\n  Summary:")
    for dev in DEVICE_ORDER:
        ratios = tbl.loc[dev]["asym/sym"]
        print(f"    {dev}: min={ratios.min():.2f}  max={ratios.max():.2f}  "
              f"median={ratios.median():.2f}  mean={ratios.mean():.2f}  std={ratios.std():.2f}")

# ARM 4-bit: check if the difference is systematic per model size
print(f"\n{'='*90}")
print("  ARM 4-bit asym/sym by model_size")
print(f"{'='*90}")
arm_4 = mnb[(mnb["device"] == "arm") & (mnb["bits"] == 4)]
tbl_arm = comparison_table(arm_4, ["model_size", "seq_length"], "symmetry", ["sym", "asym"])
tbl_arm.columns = ["sym", "asym", "asym/sym"]
for ms in MODEL_SIZE_ORDER:
    if ms not in tbl_arm.index.get_level_values(0):
        continue
    ratios = tbl_arm.loc[ms]["asym/sym"]
    mean_r = ratios.mean()
    direction = "asym SLOWER" if mean_r > 1.05 else ("asym FASTER" if mean_r < 0.95 else "~EQUAL")
    print(f"  {ms}: mean={mean_r:.2f} ({direction})  per-seq={[f'{v:.2f}' for v in ratios.values]}")


  B. MNB sym vs asym — 4-bit (ratio = asym/sym)
                                  sym     asym  asym/sym
device model_size seq_length                            
amd    0.5b       1              7.05     7.12      1.01
                  128          173.69   131.98      0.76
                  256          304.46   269.79      0.89
                  512          690.23   582.99      0.84
                  1024        1459.12  1386.95      0.95
       1.5b       1             20.75    19.17      0.92
                  128          467.04   411.74      0.88
                  256          965.10   837.89      0.87
                  512         2010.46  1775.49      0.88
                  1024        6192.57  4053.75      0.65
       3b         1             46.51    46.57      1.00
                  128         1405.70  1447.67      1.03
                  256         2793.26  2873.39      1.03
                  512         5672.13  5771.73      1.02
                  1024       12135.81 1

### Observations

1. Intel: sym ≈ asym across both bit widths. Median ratio 1.00 (4-bit) and 0.99 (8-bit) as expected
2. AMD 8-bit: median 1.02, range 0.94–1.16 as expected

3. AMD 4-bit: median 0.98, but a few outliers where asym is faster (1.5b seq=1024: 0.65, 0.5b seq=128: 0.76). Likely measurement noise worth re-running with more iterations to confirm.

4. ARM 4-bit ≥1.5b: asym is systematically ~50% slower. This is consistent across all seq lengths for each model size. Too uniform to be scheduling noise 

5. ARM 0.5b 4-bit: opposite direction (mean 0.73, asym faster). The sym seq=1 value of 29ms vs asym 10ms looks suspicious and may need a rerun.

6. ARM 8-bit: erratic ratios (0.50–2.83), no clear pattern 

Conclusion: sym vs asym is a non-factor on x86. ARM shows a real asym penalty at 4-bit for models ≥1.5b needs investigation into the ARM kernel's zero-point handling.

## C. MNB 4-bit vs 8-bit

Quantify latency difference between 4-bit and 8-bit. 4-bit should be faster at decode (less bandwidth). At prefill, compute dominates so difference should be small.

In [88]:
# Section C: MNB 4-bit vs 8-bit
# Section B showed sym ≈ asym on x86. Use sym here (cleaner ARM data).
mnb = df[(df["format_type"] == "mnb") & (df["weight_layout"] == "original")].copy()
mnb_sym = mnb[mnb["symmetry"] == "sym"]

tbl = comparison_table(
    mnb_sym,
    group_cols=["device", "model_size", "seq_length"],
    compare_col="bits",
    compare_values=[4, 8],
)
tbl.columns = ["4-bit", "8-bit", "8b/4b"]

print(f"{'='*80}")
print(f"  C. MNB 4-bit vs 8-bit — sym, original layout (ratio = 8b/4b)")
print(f"{'='*80}")
print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))

# Verify asym ratios match on x86 (ARM asym is noisy per Section B)
tbl_asym = comparison_table(
    mnb[(mnb["symmetry"] == "asym") & (mnb["device"].isin(["amd", "intel"]))],
    group_cols=["device", "model_size", "seq_length"],
    compare_col="bits",
    compare_values=[4, 8],
)
tbl_asym.columns = ["4-bit", "8-bit", "8b/4b"]
ratio_diff = (tbl_asym["8b/4b"] - tbl.loc[["amd", "intel"]]["8b/4b"]).abs()
print(f"\nAsym vs sym ratio check (x86): max diff = {ratio_diff.max():.2f}, mean = {ratio_diff.mean():.2f}")
print(f"  (AMD 1.5b outlier from Section B accounts for max diff)")

  C. MNB 4-bit vs 8-bit — sym, original layout (ratio = 8b/4b)
                                4-bit    8-bit  8b/4b
device model_size seq_length                         
amd    0.5b       1              7.05    10.11   1.43
                  128          173.69   146.08   0.84
                  256          304.46   273.11   0.90
                  512          690.23   567.94   0.82
                  1024        1459.12  1305.17   0.89
       1.5b       1             20.75    39.64   1.91
                  128          467.04   642.76   1.38
                  256          965.10  1307.94   1.36
                  512         2010.46  2883.31   1.43
                  1024        6192.57  6459.83   1.04
       3b         1             46.51    73.19   1.57
                  128         1405.70  1292.49   0.92
                  256         2793.26  2582.51   0.92
                  512         5672.13  5410.64   0.95
                  1024       12135.81 10884.73   0.90
       7b         1

### Observations

1. x86 seq=1: 8b/4b = 1.4-2.0x. Expected. Bandwidth-bound, 4-bit weights are half the size.

2. Intel seq>=128: 8b/4b = 0.88-1.08x. Bit width barely matters at prefill (compute-bound). 7b shows slight 8-bit advantage (mean 0.93x), rest are ~1.0x.

3. AMD seq>=128: 0.5b (mean 0.86x), 3b (0.92x), 7b (0.91x) all show 8-bit slightly faster. But AMD 1.5b is an outlier: 8-bit is 1.04-1.43x slower (mean 1.30x). Same anomalous 1.5b data point from Section B. Not seen on Intel (1.5b mean 1.03x). Likely a measurement issue, worth re-running.

4. ARM 0.5b-3b: 8-bit is faster than 4-bit at seq>=128 (0.48-0.92x). Stronger effect than x86. Possibly related to 4-bit nibble unpacking overhead or a less optimized ARM 4-bit kernel. Needs profiling to confirm root cause.

5. ARM 7b-8bit: 1.6-4.3x across all seq lengths. Model exceeds 16GB device RAM, causes swapping. Not usable.

6. Sym vs asym: 8b/4b ratios consistent on x86 (AMD 1.5b outlier from Section B aside). ARM asym data noisy as before.

## D. QDQ Signed vs Unsigned Asymmetric

Check if signed vs unsigned affects QDQ performance. 4-bit fuses to MatMulNBits, 8-bit does not. Expected: ~1.0x.

In [103]:
# Section D: QDQ signed vs unsigned — asymmetric, block, original layout
# 4-bit: check both fused (MatMulNBits) and unfused (DequantizeLinear) paths
# 8-bit: only fused scenario (which is actually unfused — 8-bit doesn't fuse)

qdq_asym_block_orig = df[
    (df["format_type"] == "qdq") & (df["symmetry"] == "asym") & (df["granularity"] == "block") &
    (df["weight_layout"] == "original")
].copy()

configs = [
    (4, "qdq_fused",   "4-bit fused (MatMulNBits kernel)"),
    (4, "qdq_unfused", "4-bit unfused (DequantizeLinear + MatMul)"),
    (8, "qdq_fused",   "8-bit (no fusion — runs DequantizeLinear + MatMul)"),
]

for bits, scenario, label in configs:
    sub = qdq_asym_block_orig[(qdq_asym_block_orig["bits"] == bits) & (qdq_asym_block_orig["scenario"] == scenario)]
    tbl = comparison_table(
        sub,
        group_cols=["device", "model_size", "seq_length"],
        compare_col="signedness",
        compare_values=["signed", "unsigned"],
    )
    tbl.columns = ["signed", "unsigned", "u/s"]

    print(f"\n{'='*90}")
    print(f"  D. {label} — unsigned/signed ratio")
    print(f"{'='*90}")
    for dev in DEVICE_ORDER:
        try:
            ratios = tbl.loc[dev]["u/s"]
            print(f"  {dev}: min={ratios.min():.2f}  max={ratios.max():.2f}  median={ratios.median():.2f}")
        except KeyError:
            pass
    # Show outliers
    outliers = tbl[(tbl["u/s"] < 0.85) | (tbl["u/s"] > 1.15)]
    if not outliers.empty:
        print(f"  Outliers (>15% diff):")
        print(outliers.to_string(float_format=lambda x: f"{x:.2f}"))



  D. 4-bit fused (MatMulNBits kernel) — unsigned/signed ratio
  amd: min=0.91  max=1.05  median=1.00
  intel: min=0.92  max=1.13  median=1.01
  arm: min=0.88  max=1.75  median=1.00
  Outliers (>15% diff):
                              signed  unsigned  u/s
device model_size seq_length                       
arm    0.5b       1            11.32     19.76 1.75
                  128         268.96    381.43 1.42

  D. 4-bit unfused (DequantizeLinear + MatMul) — unsigned/signed ratio
  amd: min=0.85  max=0.98  median=0.92
  intel: min=0.79  max=0.98  median=0.90
  arm: min=0.83  max=1.13  median=1.00
  Outliers (>15% diff):
                               signed  unsigned  u/s
device model_size seq_length                        
intel  0.5b       1            984.20    773.62 0.79
                  128         1275.98   1051.81 0.82
       1.5b       1           3190.45   2698.07 0.85
       7b         1          14139.84  11965.76 0.85
                  128        18216.17  15296.36 0.84


### Observations

1. 4-bit fused: signed = unsigned on x86. AMD median 1.00 (0.91-1.05). Intel median 1.01 (0.92-1.13). Same kernel, no difference as expected.

2. 4-bit unfused: unsigned consistently faster on x86. AMD median 0.92 (0.85-0.98), Intel median 0.90 (0.79-0.98). Every data point has unsigned <= signed. Effect spans all model sizes. The int4 vs uint4 nibble extraction in the scalar DQ kernel is the likely cause.

3. 8-bit: signed = unsigned on x86. AMD median 0.99, Intel median 0.99. No difference for byte-width types.

4. ARM: median ~1.0 across all three configs. Outliers in both directions, likely measurement variance and memory pressure on large models (same as Sections B and C).

Conclusion: Non-factor for the fused path. The unfused 4-bit DQ kernel runs 8-10% faster with unsigned weights on x86.


# Below section are Copilot generated: Need to polish and verify

## E. QDQ Block vs Channel vs Transpose (Weight Layout)

- **Block original**: Fuses to MatMulNBits for 4-bit
- **Channel original**: No fusion (no block structure)
- **Block transpose**: Transpose between DQ and MatMul — tests if it breaks fusion

Note: `channel-sym-signed-transpose` models fail to load (ORT bug).

In [113]:
# Section E: QDQ block vs channel vs transpose
# Focus on "qdq_fused" scenario across all layouts and granularities
# Use asym-signed as the common config across all layouts

qdq_fused = df[(df["format_type"] == "qdq") & (df["scenario"] == "qdq_fused")].copy()

# Create a composite layout label
qdq_fused["layout_label"] = qdq_fused["granularity"] + "_" + qdq_fused["weight_layout"]
# e.g. "block_original", "block_transpose", "channel_original", "channel_transpose"

for bits in [4, 8]:
    sub = qdq_fused[(qdq_fused["bits"] == bits) & (qdq_fused["symmetry"] == "asym") & (qdq_fused["signedness"] == "signed")]
    available_layouts = sorted(sub["layout_label"].unique())
    print(f"\n{'='*120}")
    print(f"  E. QDQ layout comparison — {bits}-bit asym signed, fused scenario")
    print(f"  Available layouts: {available_layouts}")
    print(f"{'='*120}")

    tbl = comparison_table(
        sub,
        group_cols=["device", "model_size", "seq_length"],
        compare_col="layout_label",
        compare_values=available_layouts,
    )
    print(tbl.to_string(float_format=lambda x: f"{x:.1f}"))



  E. QDQ layout comparison — 4-bit asym signed, fused scenario
  Available layouts: ['block_original', 'block_transpose', 'channel_original', 'channel_transpose']
                              block_original  block_transpose  channel_original  channel_transpose
device model_size seq_length                                                                      
amd    0.5b       1                      8.3           1012.9             769.0              660.6
                  128                  228.9           1131.2             911.4              856.6
                  256                  479.5           1275.9            1139.5             1022.1
                  512                  982.2           1693.2            1403.9             1394.8
                  1024                2180.5           2501.8            2153.3             2277.0
       1.5b       1                     20.6           3247.6            2421.5             2392.9
                  128                  638.4

In [114]:
# E (continued): For 4-bit, compare fused block-original against MNB native as baseline
# This shows whether the fused QDQ path truly matches native MatMulNBits

mnb_4_asym = df[(df["format_type"] == "mnb") & (df["bits"] == 4) &
                (df["symmetry"] == "asym") & (df["weight_layout"] == "original")].copy()
mnb_4_asym["layout_label"] = "mnb_native"

qdq_4_asym_signed = qdq_fused[(qdq_fused["bits"] == 4) & (qdq_fused["symmetry"] == "asym") &
                               (qdq_fused["signedness"] == "signed")]

combined_e = pd.concat([mnb_4_asym, qdq_4_asym_signed], ignore_index=True)
available = sorted(combined_e["layout_label"].unique())

print(f"\n{'='*120}")
print(f"  E (with MNB baseline). 4-bit asym — MNB native vs QDQ layouts")
print(f"  Layouts: {available}")
print(f"{'='*120}")

tbl = comparison_table(
    combined_e,
    group_cols=["device", "model_size", "seq_length"],
    compare_col="layout_label",
    compare_values=available,
)
print(tbl.to_string(float_format=lambda x: f"{x:.1f}"))



  E (with MNB baseline). 4-bit asym — MNB native vs QDQ layouts
  Layouts: ['block_original', 'block_transpose', 'channel_original', 'channel_transpose', 'mnb_native']
                              block_original  block_transpose  channel_original  channel_transpose  mnb_native
device model_size seq_length                                                                                  
amd    0.5b       1                      8.3           1012.9             769.0              660.6         7.1
                  128                  228.9           1131.2             911.4              856.6       132.0
                  256                  479.5           1275.9            1139.5             1022.1       269.8
                  512                  982.2           1693.2            1403.9             1394.8       583.0
                  1024                2180.5           2501.8            2153.3             2277.0      1387.0
       1.5b       1                     20.6          

### E — Observations

**4-bit seq=1 — block_original dominates (fused):**
- AMD: block_orig 8–79ms vs unfused layouts 661–14,857ms. **80–190× speedup from fusion.**
- Intel/ARM: Same pattern. ARM block_orig 7b (59ms) is faster than AMD (79ms) and Intel (101ms).

**Block transpose breaks fusion.** The Transpose op between DQ and MatMul prevents QDQ→MatMulNBits fusion. Performance drops to unfused levels.

**Channel original:** Never fuses (no block structure). Same slow DQ→fp32→MatMul path.

**8 channel-sym-signed-transpose models crash** at `create_session()` — ORT bug.

**Key insight:** The fused vs unfused cliff (50–200× at decode) is the dominant effect in this analysis.

## F. QDQ 4-bit vs 8-bit

4-bit block fuses to MatMulNBits; 8-bit does NOT. This reveals the cost of missing 8-bit fusion.

In [115]:
# Section F: QDQ 4-bit vs 8-bit — block, fused, original layout
qdq_block_fused_orig = df[
    (df["format_type"] == "qdq") & (df["granularity"] == "block") &
    (df["scenario"] == "qdq_fused") & (df["weight_layout"] == "original") &
    (df["symmetry"] == "asym") & (df["signedness"] == "signed")
].copy()

tbl = comparison_table(
    qdq_block_fused_orig,
    group_cols=["device", "model_size", "seq_length"],
    compare_col="bits",
    compare_values=[4, 8],
)
tbl.columns = ["4-bit", "8-bit", "8b/4b"]
print(f"\n{'='*80}")
print(f"  F. QDQ 4-bit vs 8-bit — block asym signed fused original")
print(f"  (4-bit fuses to MatMulNBits; 8-bit does NOT fuse)")
print(f"{'='*80}")
print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))

# Also compare transpose layout
qdq_block_fused_trans = df[
    (df["format_type"] == "qdq") & (df["granularity"] == "block") &
    (df["scenario"] == "qdq_fused") & (df["weight_layout"] == "transpose") &
    (df["symmetry"] == "asym") & (df["signedness"] == "signed")
].copy()

if not qdq_block_fused_trans.empty:
    tbl2 = comparison_table(
        qdq_block_fused_trans,
        group_cols=["device", "model_size", "seq_length"],
        compare_col="bits",
        compare_values=[4, 8],
    )
    tbl2.columns = ["4-bit", "8-bit", "8b/4b"]
    print(f"\n{'='*80}")
    print(f"  F (transpose). QDQ 4-bit vs 8-bit — block asym signed fused TRANSPOSE")
    print(f"{'='*80}")
    print(tbl2.to_string(float_format=lambda x: f"{x:.2f}"))



  F. QDQ 4-bit vs 8-bit — block asym signed fused original
  (4-bit fuses to MatMulNBits; 8-bit does NOT fuse)
                                4-bit    8-bit  8b/4b
device model_size seq_length                         
amd    0.5b       1              8.31   186.85  22.49
                  128          228.87   416.90   1.82
                  256          479.46   660.33   1.38
                  512          982.16  1230.04   1.25
                  1024        2180.51  2474.20   1.13
       1.5b       1             20.55   596.04  29.00
                  128          638.45  1284.75   2.01
                  256         1304.62  2020.02   1.55
                  512         2707.78  3663.19   1.35
                  1024        5731.82  7126.40   1.24
       3b         1             40.97  1210.69  29.55
                  128         1276.64  2665.21   2.09
                  256         2608.19  4396.43   1.69
                  512         5312.74  8031.21   1.51
                  1024  

### F — Observations

**Original layout (4-bit fuses, 8-bit does not):**
- 8b/4b ratio at seq=1: AMD 22–33×, Intel 25–37×, ARM 11–348×. Entirely due to missing 8-bit fusion.
- At seq=1024: converges to 1.0–1.5× on x86.

**Transpose layout (both unfused):**
- 8b/4b = 0.48–0.81× on AMD/Intel. **8-bit is actually faster** — simpler dequant (no bit unpacking).

**Implication:** An 8-bit MatMulNBits fused kernel would give 22–37× decode speedup. The transpose data shows 8-bit dequant is cheaper than 4-bit, so a fused 8-bit kernel could be very competitive.

## G. Cross-Device Consistency

Compare absolute latency and ratios across AMD, Intel, ARM. Identify device-specific kernel gaps.

In [116]:
# Section G: Cross-device comparison — MNB native decode latency (seq_length=1)
# This is the clearest signal of device-level kernel quality

decode = df[df["seq_length"] == 1].copy()

# G.1: MNB native decode across devices
mnb_decode = decode[(decode["format_type"] == "mnb") & (decode["weight_layout"] == "original")]

for bits in [4, 8]:
    for sym in ["sym", "asym"]:
        sub = mnb_decode[(mnb_decode["bits"] == bits) & (mnb_decode["symmetry"] == sym)]
        tbl = comparison_table(
            sub,
            group_cols=["model_size"],
            compare_col="device",
            compare_values=["amd", "intel", "arm"],
        )
        # Normalize to AMD
        for col in ["intel", "arm"]:
            if col in tbl.columns:
                tbl[f"{col}/amd"] = tbl[col] / tbl["amd"]

        print(f"\n{'='*90}")
        print(f"  G.1 Cross-device MNB decode — {bits}-bit {sym} (seq=1)")
        print(f"{'='*90}")
        print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))



  G.1 Cross-device MNB decode — 4-bit sym (seq=1)
             amd  intel   arm  intel/amd  arm/amd
model_size                                       
0.5b        7.05   8.21 29.08       1.17     4.12
1.5b       20.75  21.83 64.27       1.05     3.10
3b         46.51  41.24 66.90       0.89     1.44
7b         87.11  87.97 94.91       1.01     1.09

  G.1 Cross-device MNB decode — 4-bit asym (seq=1)
             amd  intel    arm  intel/amd  arm/amd
model_size                                        
0.5b        7.12   8.89   9.69       1.25     1.36
1.5b       19.17  22.92  87.89       1.20     4.59
3b         46.57  41.80 116.53       0.90     2.50
7b         98.10  92.03 111.99       0.94     1.14

  G.1 Cross-device MNB decode — 8-bit sym (seq=1)
              amd  intel    arm  intel/amd  arm/amd
model_size                                         
0.5b        10.11  15.31  39.12       1.51     3.87
1.5b        39.64  38.74 104.25       0.98     2.63
3b          73.19  79.83 148.58 

In [117]:
# G.2: Cross-device comparison for QDQ fused block 4-bit decode (seq=1)
# This tests whether the fused pathway works equally well on all devices

qdq_fused_decode = decode[
    (decode["format_type"] == "qdq") & (decode["granularity"] == "block") &
    (decode["signedness"] == "signed") & (decode["symmetry"] == "asym") &
    (decode["scenario"] == "qdq_fused")
]

for layout in ["original", "transpose"]:
    sub = qdq_fused_decode[qdq_fused_decode["weight_layout"] == layout]
    if sub.empty:
        continue
    for bits in [4, 8]:
        bits_sub = sub[sub["bits"] == bits]
        if bits_sub.empty:
            continue
        tbl = comparison_table(
            bits_sub,
            group_cols=["model_size"],
            compare_col="device",
            compare_values=["amd", "intel", "arm"],
        )
        for col in ["intel", "arm"]:
            if col in tbl.columns:
                tbl[f"{col}/amd"] = tbl[col] / tbl["amd"]

        print(f"\n{'='*90}")
        print(f"  G.2 Cross-device QDQ fused decode — {bits}-bit block asym signed {layout} (seq=1)")
        print(f"{'='*90}")
        print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))



  G.2 Cross-device QDQ fused decode — 4-bit block asym signed original (seq=1)
             amd  intel   arm  intel/amd  arm/amd
model_size                                       
0.5b        8.31   8.38 11.32       1.01     1.36
1.5b       20.55  22.72 33.55       1.11     1.63
3b         40.97  44.43 56.42       1.08     1.38
7b         78.81 100.73 59.08       1.28     0.75

  G.2 Cross-device QDQ fused decode — 8-bit block asym signed original (seq=1)
               amd   intel      arm  intel/amd  arm/amd
model_size                                             
0.5b        186.85  209.18   128.10       1.12     0.69
1.5b        596.04  710.44   551.40       1.19     0.93
3b         1210.69 1650.63  1130.80       1.36     0.93
7b         2557.51 3418.28 20546.05       1.34     8.03

  G.2 Cross-device QDQ fused decode — 4-bit block asym signed transpose (seq=1)
                amd    intel     arm  intel/amd  arm/amd
model_size                                              
0.5b     

In [118]:
# G.3: Full scaling comparison — all seq lengths for MNB 4-bit asym across devices
# Shows whether scaling behavior (seq=1 → 1024) is consistent across devices

mnb_4_asym = df[
    (df["format_type"] == "mnb") & (df["bits"] == 4) &
    (df["symmetry"] == "asym") & (df["weight_layout"] == "original")
]

tbl = comparison_table(
    mnb_4_asym,
    group_cols=["model_size", "seq_length"],
    compare_col="device",
    compare_values=["amd", "intel", "arm"],
)
for col in ["intel", "arm"]:
    if col in tbl.columns:
        tbl[f"{col}/amd"] = tbl[col] / tbl["amd"]

print(f"\n{'='*100}")
print(f"  G.3 Cross-device scaling: MNB 4-bit asym — all seq lengths")
print(f"{'='*100}")
print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))



  G.3 Cross-device scaling: MNB 4-bit asym — all seq lengths
                           amd    intel      arm  intel/amd  arm/amd
model_size seq_length                                               
0.5b       1              7.12     8.89     9.69       1.25     1.36
           128          131.98   399.82   317.20       3.03     2.40
           256          269.79   662.35   763.95       2.46     2.83
           512          582.99  1241.21  1652.43       2.13     2.83
           1024        1386.95  2982.13  3390.14       2.15     2.44
1.5b       1             19.17    22.92    87.89       1.20     4.59
           128          411.74   968.97  1331.42       2.35     3.23
           256          837.89  1870.20  2353.81       2.23     2.81
           512         1775.49  3793.80  4750.05       2.14     2.68
           1024        4053.75  7873.24  8388.75       1.94     2.07
3b         1             46.57    41.80   116.53       0.90     2.50
           128         1447.67  1835.56  

### G — Observations

**AMD vs Intel:**
- seq=1: Intel/AMD ≈ 0.90–1.25×. Same x86 kernels.
- seq=1024: Intel/AMD = 1.19–2.15×. AMD has better prefill throughput (AVX-512/bandwidth advantage).

**ARM vs AMD (MNB 4-bit):**
- seq=1: ARM/AMD = 1.14–4.59×. ARM 1.5b is extreme outlier (4.6×).
- ARM 7b QDQ fused is 0.75× AMD — ARM actually faster for fused 7b decode.

**Pattern:** ARM's weakness is MNB native decode specifically, not QDQ fused. The ARM QDQ fused path has a strong kernel that even beats x86 for 7b.

## H. Model Load Time

Session creation time across formats/layouts/devices. Includes graph optimization + weight preparation.

In [119]:
# Section H: Model load time
# One load time per model (not per seq_length), so deduplicate
load_times = df.drop_duplicates(
    subset=["device", "format_type", "model_size", "bits", "symmetry",
            "granularity", "signedness", "weight_layout", "scenario"]
)[["device", "format_type", "model_size", "bits", "symmetry", "granularity",
   "signedness", "weight_layout", "scenario", "model_load_time_s"]].copy()

load_times = load_times.sort_values(["device", "format_type", "weight_layout", "bits", "model_size"])

# H.1: MNB load times
mnb_load = load_times[load_times["format_type"] == "mnb"]
tbl = mnb_load.pivot_table(
    index=["bits", "symmetry", "model_size"],
    columns="device",
    values="model_load_time_s"
)
print(f"\n{'='*80}")
print(f"  H.1 MNB model load times (seconds)")
print(f"{'='*80}")
print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))

# H.2: QDQ load times — focus on block fused, original vs transpose
qdq_load = load_times[
    (load_times["format_type"] == "qdq") & (load_times["granularity"] == "block") &
    (load_times["scenario"] == "qdq_fused") & (load_times["signedness"] == "signed") &
    (load_times["symmetry"] == "asym")
]
tbl2 = qdq_load.pivot_table(
    index=["bits", "weight_layout", "model_size"],
    columns="device",
    values="model_load_time_s"
)
print(f"\n{'='*80}")
print(f"  H.2 QDQ block asym signed fused — load times (seconds)")
print(f"{'='*80}")
print(tbl2.to_string(float_format=lambda x: f"{x:.2f}"))

# H.3: Flag extreme load times (>30s)
extreme = load_times[load_times["model_load_time_s"] > 30]
if not extreme.empty:
    print(f"\n{'='*80}")
    print(f"  H.3 EXTREME load times (>30s) — possible issues")
    print(f"{'='*80}")
    print(extreme.to_string(index=False, float_format=lambda x: f"{x:.1f}"))
else:
    print("\nNo extreme load times (>30s) detected.")



  H.1 MNB model load times (seconds)
device                     amd   arm  intel
bits symmetry model_size                   
4    asym     0.5b        0.78  0.49   1.32
              1.5b        1.83  3.58   2.25
              3b          5.90  6.46   4.56
              7b          9.97 12.76   9.76
     sym      0.5b        0.85  1.85   1.14
              1.5b        2.15  4.56   2.86
              3b          6.86  8.05   5.70
              7b          9.47 18.42  12.35
8    asym     0.5b        0.87  3.15   1.17
              1.5b        3.84  6.58   2.82
              3b          6.85 12.17   6.01
              7b         12.64 48.39  13.48
     sym      0.5b        0.97  2.84   1.27
              1.5b        4.20  8.07   3.43
              3b          7.40 13.65   6.96
              7b         13.28 51.37  15.62

  H.2 QDQ block asym signed fused — load times (seconds)
device                          amd   arm  intel
bits weight_layout model_size                   
4    original 

### H — Observations

- QDQ fused load ≈ 1.0–1.4× MNB load. Slight overhead from graph rewriting.
- Transpose loads are fast (<0.5s) — no fusion rewriting.
- **Channel transpose load is catastrophic:** 40–228s. Pathological graph optimization — ORT bug.
- ARM MNB 7b-8bit: 48–51s load — memory pressure.

## I. Unfused Overhead Scaling (seq_length dependency)

Is DQ overhead constant (full weight dequant each time) or does it scale with seq_length? For weight-only quantization, DQ should be constant.

In [120]:
# Section I: Unfused overhead scaling
# Compare unfused/fused ratio across seq_lengths — if DQ cost is constant,
# the ratio should decrease as seq_length grows (DQ is amortized)

fused_4 = df[
    (df["format_type"] == "qdq") & (df["bits"] == 4) & (df["granularity"] == "block") &
    (df["weight_layout"] == "original") & (df["scenario"] == "qdq_fused") &
    (df["signedness"] == "signed") & (df["symmetry"] == "asym")
][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "fused_ms"})

unfused_4 = df[
    (df["format_type"] == "qdq") & (df["bits"] == 4) & (df["granularity"] == "block") &
    (df["weight_layout"] == "original") & (df["scenario"] == "qdq_unfused") &
    (df["signedness"] == "signed") & (df["symmetry"] == "asym")
][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "unfused_ms"})

scaling = fused_4.merge(unfused_4, on=["device", "model_size", "seq_length"])
scaling["unfused/fused"] = scaling["unfused_ms"] / scaling["fused_ms"]
scaling["overhead_ms"] = scaling["unfused_ms"] - scaling["fused_ms"]

print(f"\n{'='*100}")
print(f"  I. Unfused overhead scaling — 4-bit block asym signed")
print(f"  If DQ cost is constant, overhead_ms should be ~constant and unfused/fused ratio should DECREASE")
print(f"{'='*100}")
scaling_display = scaling.set_index(["device", "model_size", "seq_length"]).sort_index()
print(scaling_display.to_string(float_format=lambda x: f"{x:.1f}"))



  I. Unfused overhead scaling — 4-bit block asym signed
  If DQ cost is constant, overhead_ms should be ~constant and unfused/fused ratio should DECREASE
                              fused_ms  unfused_ms  unfused/fused  overhead_ms
device model_size seq_length                                                  
amd    0.5b       1                8.3       691.9           83.3        683.6
                  128            228.9       934.1            4.1        705.3
                  256            479.5      1200.6            2.5        721.1
                  512            982.2      1786.4            1.8        804.3
                  1024          2180.5      3007.3            1.4        826.8
       1.5b       1               20.6      2267.2          110.3       2246.6
                  128            638.4      2991.6            4.7       2353.2
                  256           1304.6      3672.5            2.8       2367.8
                  512           2707.8      5244.9     

### I — Observations

DQ overhead is NOT purely constant — varies by device:

- **AMD:** Overhead grows 1.2–1.9× from seq=1→1024. fp32 MatMul memory pressure increases.
- **Intel:** Overhead flat or shrinks (0.7–1.1×). DQ cost dominates consistently.
- **ARM 7b:** Explodes 2.5× at seq=1024 — memory bandwidth saturation on 16GB device.

DQ itself is constant (full weight matrix each time), but the materialized fp32 MatMul stresses cache/bandwidth at longer sequences on AMD and ARM.

## J. Data Validation

In [121]:
# Section J: Data validation

print("="*80)
print("  J.1 Row counts per device")
print("="*80)
print(df.groupby("device").size().to_string())

print(f"\n{'='*80}")
print("  J.2 Breakdown by device × format_type × weight_layout × scenario")
print("="*80)
breakdown = df.groupby(["device", "format_type", "weight_layout", "scenario"]).size().unstack(fill_value=0)
print(breakdown.to_string())

print(f"\n{'='*80}")
print("  J.3 Configs per device (unique format+size+bits+sym+gran+sign+layout+scenario)")
print("="*80)
for device in DEVICE_ORDER:
    dev_df = df[df["device"] == device]
    configs = dev_df.drop_duplicates(
        subset=["format_type", "model_size", "bits", "symmetry", "granularity",
                "signedness", "weight_layout", "scenario"]
    )
    print(f"  {device}: {len(configs)} unique configs, {len(dev_df)} total rows")

# J.4: Check for duplicates after disambiguation
print(f"\n{'='*80}")
print("  J.4 Duplicate check (same device+config+seq_length)")
print("="*80)
group_cols = ["device", "format_type", "model_size", "bits", "symmetry",
              "granularity", "signedness", "weight_layout", "scenario", "seq_length"]
dupes = df.groupby(group_cols).size()
dupes_gt1 = dupes[dupes > 1]
if dupes_gt1.empty:
    print("  No duplicates found — data is clean!")
else:
    print(f"  {len(dupes_gt1)} duplicate groups found:")
    print(dupes_gt1.to_string())

# J.5: Missing configs across devices
print(f"\n{'='*80}")
print("  J.5 Configs present in one device but missing from another (seq=1 only)")
print("="*80)
config_cols = ["format_type", "model_size", "bits", "symmetry", "granularity",
               "signedness", "weight_layout", "scenario"]
decode_df = df[df["seq_length"] == 1]
for d1 in DEVICE_ORDER:
    for d2 in DEVICE_ORDER:
        if d1 >= d2:
            continue
        s1 = set(decode_df[decode_df["device"] == d1][config_cols].apply(tuple, axis=1))
        s2 = set(decode_df[decode_df["device"] == d2][config_cols].apply(tuple, axis=1))
        only_d1 = s1 - s2
        only_d2 = s2 - s1
        if only_d1 or only_d2:
            print(f"\n  {d1} vs {d2}:")
            if only_d1:
                print(f"    Only in {d1} ({len(only_d1)}):", list(only_d1)[:5], "..." if len(only_d1) > 5 else "")
            if only_d2:
                print(f"    Only in {d2} ({len(only_d2)}):", list(only_d2)[:5], "..." if len(only_d2) > 5 else "")


  J.1 Row counts per device
device
amd      640
arm      560
intel    640

  J.2 Breakdown by device × format_type × weight_layout × scenario
scenario                          native  qdq_fused  qdq_unfused
device format_type weight_layout                                
amd    mnb         original           80          0            0
       qdq         original            0        240           60
                   transpose           0        200           60
arm    mnb         original           80          0            0
       qdq         original            0        240           60
                   transpose           0        180            0
intel  mnb         original           80          0            0
       qdq         original            0        240           60
                   transpose           0        200           60

  J.3 Configs per device (unique format+size+bits+sym+gran+sign+layout+scenario)
  amd: 128 unique configs, 640 total rows
  intel: 128 unique

## Actionable Findings

| # | Finding | Impact | Action |
|---|---------|--------|--------|
| 1 | **8-bit QDQ has no fused kernel** → 22–37× slower decode than 4-bit | HIGH | Implement 8-bit MatMulNBits fusion. Transpose data shows 8-bit dequant is cheaper — fused 8-bit kernel could be very fast. |
| 2 | **ARM MNB native slower than QDQ fused** for ≥1.5b decode | MEDIUM | Investigate ARM MNB graph construction vs QDQ-fused path. |
| 3 | **8 channel-sym-signed-transpose models crash** at create_session | MEDIUM | File ORT bug. |
| 4 | **Channel transpose load: 40–228s** | MEDIUM | File ORT bug — pathological graph optimization. |
| 5 | **Block transpose breaks fusion** → 80–190× slower | LOW (by design) | Document: transpose between DQ and MatMul prevents fusion. Use original layout for 4-bit block. |
| 6 | Sym/asym and signed/unsigned are non-factors on x86 | INFO | Model producers have full freedom on these choices. |